# Module 5: file health with Delta Analyzer

One table, one command.

Delta Analyzer reads the Delta log and the Parquet files directly and reports what is actually on disk: how many files, how many row groups, and the compressed size of every column.

**Why not VertiPaq Analyzer?** On a Direct Lake model it reports 0 bytes for every column, because the size columns are disabled upstream in OneLake. Delta Analyzer goes to the files instead, so it is the only way to see what a column really costs in a Direct Lake world.

> Run as a **PySpark** notebook. On the 3M row `Sales` table this takes a minute or two.

## 1. Install Semantic Link Labs

In [ ]:
%pip install -q semantic-link-labs

## 2. Choose the table

Defaults to `Sales` in your own `workshop` lakehouse, which is the same table your models read.

The others you have: `Date`, `Product`, `Customer`, `Territory`, `UserAccess`, `Sales_messy`, `Product_messy`.

In [ ]:
import sempy_labs as labs

LAKEHOUSE = "workshop"
TABLE = "Sales"

# PRESENTER: the 100M row copies, for the V-Order half of the module.
# LAKEHOUSE = "workshop_shared"
# TABLE = "sales_xl_vorder"    # or sales_xl_raw / sales_xl_compact / sales_xl_gold

# Cardinality is the WHY behind every column size here, and on 3M rows it is
# cheap. Set it True on the 100M row copies unless you pre-ran the night before.
SKIP_CARDINALITY = False

## 3. Run Delta Analyzer

Open the **Columns** tab in the result and sort by **Compressed Size**. Read it top and bottom, not left to right.

In [ ]:
_ = labs.delta_analyzer(
    TABLE,
    lakehouse=LAKEHOUSE,
    skip_cardinality=SKIP_CARDINALITY, approx_distinct_count=True)

## Talking points: `Sales`

**Put Cardinality next to Compressed Size and the whole module is on one screen.** The expensive columns are the high-cardinality ones. Not the widest, not the ones with the longest names. The ones with the most distinct values.

**Find `NetAmount`, `NetAmount_Whole` and `NetAmount_Frac`.** They hold the same information. `NetAmount` has roughly 2.2M distinct values; the other two have about 1,600 and 10,000. Compare what each costs on disk. That is Module 4's column-splitting argument, in bytes, on your own table.

**Now find the key columns.** A surrogate key is distinct on every row, so there is no repetition for any encoding to find. That is the argument for not storing keys nobody queries.

**Files and row groups matter too.** The same bytes spread across many small files cost more to read than a few large ones, because the work is paid per row group and per column chunk, not per byte.

## Talking points: the 100M row copies (presenter)

**File count first.** 24 files, not 400. Compaction is the biggest lever in this module and the one people least expect: 400 files down to 24 took this table from 2,934 MB to 2,401 MB, and cold load from 4m33s to 56s.

**Bottom of the Columns tab: `Quantity` and `Discount` cost almost nothing.** V-Order reordered the rows as it wrote the files, and these two, at 10 and 5 distinct values, ended up in near-perfect runs. They are also the columns the favoured query filters, which is the entire reason that query is quick.

**Top of the Columns tab: `SalesKey` and `OrderNumber` are the expensive ones,** and under V-Order they got *bigger*. They are big for different reasons and only one of them is fair:

- `SalesKey` is a row id, 100M distinct over 100M rows. There is no repetition for any sort order to find.
- `OrderNumber` is 5M distinct over 100M rows, roughly 20 lines per order, so there **is** repetition to find. It got worse because the reordering scrambled the order it arrived in.

**That last pair is also the evidence.** A change of encoding or compression alone could not make two columns collapse to nothing while making two others worse. Only physically reordering the rows does that. If someone asks how you know V-Order sorts, this table is the answer.

**The trade, said out loud.** V-Order makes the columns you filter cheap and charges you on the keys you do not. Sort by what you filter.

## 4. Hand the Spark session back

Run this when you have finished looking at the results. Delta Analyzer needs Spark, and an idle session holds capacity for 30 minutes. With a room full of people that is a lot of nothing happening, and it is what makes everyone else's next run slow.

Nothing runs after this cell.


In [ ]:
# detach=False matters. detach defaults to True, and on a high-concurrency
# session the default only detaches this notebook and leaves the session up.
try:
    notebookutils.session.stop(detach=False)
except TypeError:
    notebookutils.session.stop()        # older runtimes have no detach parameter
except Exception as e:
    print(f"Could not stop the session ({e}). Use Stop session in the toolbar.")
